In [ ]:
pip install -r requirements.txt

In [ ]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)


locations = [
    {"name": "Ringsted", "latitude": 56.1697, "longitude": 9.5451},
    {"name": "Silkeborg", "latitude": 55.4426, "longitude": 11.7901},
    {"name": "Danmark", "latitude": 56, "longitude": 10}]


# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
params = {
	"latitude": [56.1697, 55.4426, 56],
	"longitude": [9.5451, 11.7901, 10],
	"start_date": "2022-01-01",
	"end_date": "2024-12-31",
	"hourly": ["wind_speed_10m", "wind_speed_80m", "wind_speed_120m", "wind_speed_180m", "wind_direction_10m", "wind_direction_80m", "wind_direction_120m", "wind_direction_180m", "uv_index", "sunshine_duration"],
}
responses = openmeteo.weather_api(url, params = params)

# Process 3 locations
for i, response in enumerate(responses):
	location = locations[i]
	print(f"\nLocation: {location['name']}")
	assert abs(response.Latitude() - location["latitude"]) < 0.01, "Latitude does not match"
	assert abs(response.Longitude() - location["longitude"]) < 0.01, "Longitude does not match"

	print(f"\nCoordinates: {response.Latitude()}°N {response.Longitude()}°E")
	print(f"Elevation: {response.Elevation()} m asl")
	print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")
	
	# Process hourly data. The order of variables needs to be the same as requested.
	hourly = response.Hourly()
	hourly_wind_speed_10m = hourly.Variables(0).ValuesAsNumpy()
	hourly_wind_speed_80m = hourly.Variables(1).ValuesAsNumpy()
	hourly_wind_speed_120m = hourly.Variables(2).ValuesAsNumpy()
	hourly_wind_speed_180m = hourly.Variables(3).ValuesAsNumpy()
	hourly_wind_direction_10m = hourly.Variables(4).ValuesAsNumpy()
	hourly_wind_direction_80m = hourly.Variables(5).ValuesAsNumpy()
	hourly_wind_direction_120m = hourly.Variables(6).ValuesAsNumpy()
	hourly_wind_direction_180m = hourly.Variables(7).ValuesAsNumpy()
	hourly_uv_index = hourly.Variables(8).ValuesAsNumpy()
	hourly_sunshine_duration = hourly.Variables(9).ValuesAsNumpy()
	
	hourly_data = {"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)}
	
	hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
	hourly_data["wind_speed_80m"] = hourly_wind_speed_80m
	hourly_data["wind_speed_120m"] = hourly_wind_speed_120m
	hourly_data["wind_speed_180m"] = hourly_wind_speed_180m
	hourly_data["wind_direction_10m"] = hourly_wind_direction_10m
	hourly_data["wind_direction_80m"] = hourly_wind_direction_80m
	hourly_data["wind_direction_120m"] = hourly_wind_direction_120m
	hourly_data["wind_direction_180m"] = hourly_wind_direction_180m
	hourly_data["uv_index"] = hourly_uv_index
	hourly_data["sunshine_duration"] = hourly_sunshine_duration
	
	hourly_dataframe = pd.DataFrame(data = hourly_data)
	print("\nHourly data\n", hourly_dataframe)
	hourly_dataframe.to_csv(f"hourly_data_{location['name']}_{response.Latitude()}_{response.Longitude()}.csv", index = False)
	